# PNGF on Google Colab
**Precomputed Numerical Green Function — EM Inverse Design**

## ⚠️ Before running anything:
1. Go to **Runtime → Change runtime type**
2. Set **Hardware accelerator** to `None` and **Runtime shape** to `High-RAM`
3. Click **Save** — the `acmefdfd` precomputation step needs ~50 GB RAM

## Workflow
| Step | Cell | Time estimate |
|------|------|---------------|
| Install dependencies | §1 | ~3 min |
| Mount Google Drive (optional) | §2 | 30 sec |
| Clone repo | §3 | 10 sec |
| Build `acmefdfd` | §4 | ~1 min |
| Build `pngf-opt` | §5 | ~30 sec |
| SSH tunnel for Cursor | §6 | 30 sec |
| Run `acmefdfd` (precompute G matrices) | §7 | **~1–4 hours, 50 GB RAM** |
| Run `pngf-opt` (optimization) | §8 | ~minutes |

In [ ]:
# ── §0  Check available RAM ──────────────────────────────────────────────────
import subprocess
result = subprocess.run(['cat', '/proc/meminfo'], capture_output=True, text=True)
for line in result.stdout.splitlines():
    if line.startswith('MemTotal'):
        gb = int(line.split()[1]) / 1e6
        symbol = '✅' if gb >= 40 else '❌'
        print(f"{symbol}  Total RAM: {gb:.1f} GB")
        if gb < 40:
            print("   → Go to Runtime → Change runtime type → High-RAM and reconnect.")
        break

In [ ]:
# ── §1  Install system dependencies ─────────────────────────────────────────
# All packages come from Ubuntu apt — no manual compilation of MUMPS needed.
!apt-get update -qq
!apt-get install -y -qq \
    gcc g++ gfortran \
    libopenmpi-dev openmpi-bin \
    libeigen3-dev \
    libmetis-dev \
    libscalapack-openmpi-dev \
    libmumps-dev \
    libopenblas-dev \
    liblapack-dev
print("✅ All dependencies installed.")

In [ ]:
# ── §2  Mount Google Drive (recommended — persists Gmat files across sessions)
# Skip this cell if you don't want to use Drive.
from google.colab import drive
drive.mount('/content/drive')
GDRIVE_PNGF = '/content/drive/MyDrive/PNGF_outputs'
import os; os.makedirs(GDRIVE_PNGF, exist_ok=True)
print(f"✅ Drive mounted. Gmat files will be saved to:\n   {GDRIVE_PNGF}")

In [ ]:
# ── §3  Clone repository ─────────────────────────────────────────────────────
import os
REPO_DIR = '/root/PNGF'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/ACME-Lab-Stanford/PNGF.git {REPO_DIR}
else:
    print("Repo already present — pulling latest changes.")
    !git -C {REPO_DIR} pull
print("\n✅ Repo ready at", REPO_DIR)
!ls {REPO_DIR}

In [ ]:
# ── §4  Build acmefdfd ───────────────────────────────────────────────────────
# The Makefile auto-detects Linux and uses mpic++ + apt-installed MUMPS/Eigen.
%cd /root/PNGF/acmefdfd
!make clean && make
import os
if os.path.isfile('acmefdfd'):
    print("\n✅ acmefdfd built successfully.")
else:
    print("\n❌ Build failed — check output above.")

In [ ]:
# ── §5  Build pngf-opt ───────────────────────────────────────────────────────
# Uses g++ + OpenBLAS on Linux (no MKL needed).
%cd /root/PNGF/pngf-opt
!make clean && make
import os
if os.path.isfile('pngf-opt'):
    print("\n✅ pngf-opt built successfully.")
else:
    print("\n❌ Build failed — check output above.")

In [ ]:
# ── §6  SSH tunnel — connect Cursor to this Colab VM ────────────────────────
#
# Requires a FREE ngrok account:
#   1. Sign up at https://ngrok.com (free)
#   2. Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
#   3. Paste it below
#
NGROK_TOKEN = ""          # ← paste your ngrok authtoken here
SSH_PASSWORD = "pngf-colab"  # ← change to something you'll remember

if not NGROK_TOKEN:
    print("⚠️  Set NGROK_TOKEN above first, then re-run this cell.")
else:
    import os, time

    # Install and configure SSH server
    !apt-get install -y -qq openssh-server > /dev/null 2>&1
    !mkdir -p /run/sshd
    os.system(f'echo "root:{SSH_PASSWORD}" | chpasswd')
    !sed -i 's/#PermitRootLogin prohibit-password/PermitRootLogin yes/' /etc/ssh/sshd_config
    !grep -q 'PasswordAuthentication yes' /etc/ssh/sshd_config || echo 'PasswordAuthentication yes' >> /etc/ssh/sshd_config
    !/usr/sbin/sshd

    # Start ngrok TCP tunnel
    !pip install -q pyngrok
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    tunnel = ngrok.connect(22, "tcp")
    addr  = tunnel.public_url.replace("tcp://", "")
    host, port = addr.split(":")

    print(f"""
{'='*60}
✅  SSH tunnel active!

1. On your Mac, add this block to ~/.ssh/config:

   Host colab-pngf
     HostName {host}
     User root
     Port {port}

2. In Cursor:
   Cmd+Shift+P → "Remote-SSH: Connect to Host" → "colab-pngf"
   Password: {SSH_PASSWORD}
   Open folder: /root/PNGF

3. Or from terminal:
   ssh root@{host} -p {port}
{'='*60}
""")

In [ ]:
# ── §7  Run acmefdfd — generate G matrices ───────────────────────────────────
# ⚠️  Requires the High-RAM runtime (~50 GB peak).  Runtime: ~1–4 hours.
# Runs 5 frequencies: 25, 27.5, 30, 32.5, 35 GHz.
#
# Output files go to pngf-opt/ by default (as specified in run_acmefdfd.sh).
# If you mounted Google Drive in §2, they will also be copied there.

import os, shutil

%cd /root/PNGF/acmefdfd
!chmod +x run_acmefdfd.sh
!./run_acmefdfd.sh

# Optionally copy Gmat files to Google Drive for persistence
GDRIVE_PNGF = '/content/drive/MyDrive/PNGF_outputs'
if os.path.isdir(GDRIVE_PNGF):
    gmat_files = [f for f in os.listdir('../pngf-opt/') if f.startswith('Gmat')]
    for f in gmat_files:
        shutil.copy(f'../pngf-opt/{f}', f'{GDRIVE_PNGF}/{f}')
    print(f"✅ Copied {len(gmat_files)} Gmat file(s) to Google Drive.")

print("\n✅ acmefdfd complete.")

In [ ]:
# ── §8  Run pngf-opt — inverse design optimization ───────────────────────────
# Lightweight — runs on any Colab tier.
# Reads Gmat_sub_01.bin … Gmat_sub_05.bin from the same directory.
# Prints the initial and final antenna tile layout (ASCII art).

import os, shutil

# If Gmat files are on Drive but not locally, copy them back
GDRIVE_PNGF = '/content/drive/MyDrive/PNGF_outputs'
if os.path.isdir(GDRIVE_PNGF):
    for f in os.listdir(GDRIVE_PNGF):
        if f.startswith('Gmat') and not os.path.isfile(f'/root/PNGF/pngf-opt/{f}'):
            shutil.copy(f'{GDRIVE_PNGF}/{f}', f'/root/PNGF/pngf-opt/{f}')
            print(f'Copied {f} from Drive.')

%cd /root/PNGF/pngf-opt
!./pngf-opt